# Many-Body Chern Number on the Flux Torus

**Abstract.** This notebook generalises the single-particle Chern number to a *many-body* ground state. Threading Aharonov–Bohm fluxes $\boldsymbol\theta=(\theta_1,\theta_2)$ through a torus turns the ground state into a function on the flux torus $[0,1]^2$; its Chern number — computed with the Fukui–Hatsugai–Suzuki method applied to Slater determinants via the package's `many_body_Chern_number_Fukui_Hatsugai_Suzuki` — equals the sum of the single-particle Chern numbers of the occupied bands. We verify this for the Haldane model at half-filling and full filling.

**References**

- N. W. Ashcroft and N. D. Mermin, *Solid State Physics* (Saunders College Publishing, 1976).
- D. N. Sheng et al., Phys. Rev. Lett. **107**, 146803 (2011).
- K. Sun, Z. Gu, H. Katsura, and S. Das Sarma, Phys. Rev. Lett. **106**, 236803 (2011).
- T. Fukui, Y. Hatsugai, and H. Suzuki, J. Phys. Soc. Jpn. **74**, 1674 (2005).
- R. Resta, Rev. Mod. Phys. **66**, 899 (1994).

> **Execution note.** Authored without `nbconvert`, so cells ship *unexecuted but correct* (this notebook produces only printed Chern numbers, no figures).


## 1. The ground state as a function on the flux torus

Place the finite lattice on a torus and thread fluxes $\Phi_d = 2\pi\theta_d$ through its two handles. As derived in the lattice notebook, a hopping crossing direction $d$ with winding number $w_d$ acquires the phase

\begin{equation}
t_{ij} \mapsto t_{ij}\,\exp\!\bigl(i\,2\pi\,\theta_d\,w_d\bigr),
\qquad \theta_d \in [0,1).
\end{equation}

The real-space Hamiltonian $H(\boldsymbol\theta)$ — built by `build_real_space_tb_Hamiltonain(tb, twisted_phases_over_2π=θ)` — is thus a family parametrised by $\boldsymbol\theta$. Its many-body ground state (for a fixed number $N_{\mathrm{occ}}$ of electrons) is a Slater determinant of the $N_{\mathrm{occ}}$ lowest single-particle states,

\begin{equation}
|\Psi(\boldsymbol\theta)\rangle = c^\dagger_{\varepsilon_1(\boldsymbol\theta)}\cdots c^\dagger_{\varepsilon_{N_{\mathrm{occ}}}(\boldsymbol\theta)}|0\rangle .
\end{equation}

Because $\boldsymbol\theta\to\boldsymbol\theta+(1,0)$ or $+(0,1)$ is a pure gauge transformation, the ground state is *periodic up to a phase* on the torus $[0,1]^2$. The winding of that phase is the **many-body Chern number**

\begin{equation}
C_{\mathrm{mb}} = \frac{1}{2\pi}\int_{[0,1]^2}\Omega_{\mathrm{mb}}(\theta_1,\theta_2)\,d\theta_1 d\theta_2 \in \mathbb Z .
\end{equation}

> **Laughlin's pump.** Adiabatically advancing $\theta_1$ by $2\pi$ drives an electromotive force along the open direction; through the Hall response a net charge $\Delta Q = C_{\mathrm{mb}}$ (in units of $e$) is pumped across a cut (Sheng et al., PRL **107**, 146803 (2011)). The many-body Chern number is therefore the *same* topological response as the Hall conductance, but computed from a finite many-body wavefunction rather than a Bloch band.


## 2. FHS for Slater determinants

The Fukui–Hatsugai–Suzuki discretisation carries over verbatim, with the single-particle overlap replaced by the **overlap of two Slater determinants**. If $V^{(1)}$ and $V^{(2)}$ are $n_{\mathrm{site}}\times n_{\mathrm{occ}}$ matrices whose columns are the occupied single-particle states at two nearby flux points, the overlap matrix is

\begin{equation}
S_{ab} = \langle \psi^{(1)}_a | \psi^{(2)}_b \rangle,
\qquad
S = V^{(1)\dagger} V^{(2)},
\end{equation}

and the many-body overlap is its determinant,

\begin{equation}
\langle \Psi^{(1)} | \Psi^{(2)} \rangle = \det S .
\end{equation}

The link variable on the flux torus is the phase of this overlap,

\begin{equation}
U_\mu(\boldsymbol\theta) =
\frac{\det S(\boldsymbol\theta,\boldsymbol\theta+\hat\mu)}
{|\det S(\boldsymbol\theta,\boldsymbol\theta+\hat\mu)|},
\end{equation}

and the many-body Chern number is the plaquette-flux sum over an $n_\theta\times n_\theta$ mesh of $\boldsymbol\theta$. For a **non-interacting** system this exactly equals $\sum_{n\in\mathrm{occ}}C_n$, the sum of the single-particle Chern numbers of the occupied bands.


## 3. The Haldane model at half and full filling

We use the same directed templates as the topology notebook, on a $4\times4$ honeycomb torus ($n_{\mathrm{cell}}=16$, $n_{\mathrm{site}}=32$). The gapped lower band has $C_1 = -1$, so

- **half-filling** ($n_{\mathrm{occ}} = n_{\mathrm{cell}} = 16$): fill the lower band → $C_{\mathrm{mb}} = -1$;
- **full filling** ($n_{\mathrm{occ}} = 2n_{\mathrm{cell}} = 32$): all bands filled → $C_{\mathrm{mb}} = 0$ (a trivial atomic insulator).


In [ ]:
import numpy as np
from tightbinding_py import *


def build_haldane_model(M, phi, t1=-1.0, t2=-0.24, sample_size=(4, 4)):
    lat = initialize_real_space_lattice(
        sample_size=list(sample_size), lattice_name="honeycomb", pbc_indicator=[True, True],
    )
    tb = initialize_real_space_tightbinding_model(lat, model_name="haldane")
    add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 2)), t1))
    add_hopping_term(tb, ((((0, 0), 1), ((0, -1), 2)), t1))
    add_hopping_term(tb, ((((0, 0), 1), ((-1, 0), 2)), t1))
    for src in [1, 2]:
        sgn = 1 if src == 1 else -1
        add_hopping_term(tb, ((((0, 0), src), ((1, 0), src)), t2 * np.exp(sgn * 1j * phi)))
        add_hopping_term(tb, ((((0, 0), src), ((0, 1), src)), t2 * np.exp(-sgn * 1j * phi)))
        add_hopping_term(tb, ((((0, 0), src), ((-1, 1), src)), t2 * np.exp(sgn * 1j * phi)))
    add_hopping_term(tb, ((((0, 0), 1), ((0, 0), 1)), M), is_hermitian=False)
    add_hopping_term(tb, ((((0, 0), 2), ((0, 0), 2)), -M), is_hermitian=False)
    return tb


tb = build_haldane_model(M=0.7, phi=np.pi / 2)
n_cell = tb.lattice.n_cell
print("n_site =", tb.lattice.n_site, "  n_cell =", n_cell)

# Sanity: H(0) is Hermitian and gapped at the Fermi level.
H0 = build_real_space_tb_Hamiltonain(tb).toarray()
print("Hermitian error of H(0):", np.max(np.abs(H0 - H0.conj().T)))
Es = np.sort(np.linalg.eigvalsh(H0))
gap = Es[n_cell] - Es[n_cell - 1]
print(f"many-body gap at half filling (E_{n_cell+1} - E_{n_cell}) =", gap)

for nθ in (9, 11, 15):
    C_half = many_body_Chern_number_Fukui_Hatsugai_Suzuki(tb, n_occ=n_cell, nθ=nθ)
    C_full = many_body_Chern_number_Fukui_Hatsugai_Suzuki(tb, n_occ=2 * n_cell, nθ=nθ)
    print(f"nθ={nθ:2d}:  C_mb(half)={C_half:+.4f}   C_mb(full)={C_full:+.4f}")


The many-body Chern number at half-filling is quantised to $-1$ on even a $9\times9$ flux mesh, and full filling gives $0$, exactly as the single-particle result predicts. The gap $\approx 1$ between the $16$th and $17$th single-particle levels keeps the ground state (and hence the Slater determinant) smooth across the whole flux torus.


## 4. Relation to the single-particle Chern numbers

For a non-interacting system the many-body Chern number is *additive*: it equals the sum of the Chern numbers of the occupied single-particle bands. We check this explicitly with the single-particle integrator.


In [ ]:
Hk = build_Hk_crys(tb)
C1 = Chern_number_Fukui_Hatsugai_Suzuki(Hk, band=1, nk=31)
C2 = Chern_number_Fukui_Hatsugai_Suzuki(Hk, band=2, nk=31)
print("single-particle:  C1 =", C1, "  C2 =", C2, "  C1 + C2 =", C1 + C2)
print("many-body  half-filling C_mb =", many_body_Chern_number_Fukui_Hatsugai_Suzuki(tb, n_occ=n_cell, nθ=15))
print("many-body  full-filling C_mb =", many_body_Chern_number_Fukui_Hatsugai_Suzuki(tb, n_occ=2 * n_cell, nθ=15))


The identity $C_{\mathrm{mb}}=\sum_{n\in\mathrm{occ}}C_n$ is confirmed: half-filling matches the lower-band Chern number $C_1=-1$, and full filling matches $C_1+C_2=0$. This additivity is why the many-body construction is a *robust* way to probe band topology without ever constructing a smooth Bloch gauge — one only needs the finite real-space Hamiltonian and exact diagonalisation.

## 5. Finite-size and mesh remarks

- **Flux-mesh convergence.** The FHS sum over the flux torus converges to the integer as fast as its single-particle counterpart: $n_\theta=9$ is already exact for the gapped Haldane model above. Near a closing gap, convergence degrades and the integer emerges only for larger $n_\theta$ or larger real-space samples.
- **Real-space size.** The many-body gap $\Delta = E_{n_{\mathrm{occ}}+1}-E_{n_{\mathrm{occ}}}$ must stay open for *every* $\boldsymbol\theta$ on the torus. Small samples raise the minimal gap (fewer $\mathbf k$ points to hit the band minimum), but too small a sample distorts the edge physics; $4\times4$ is a comfortable compromise here.
- **Degenerate ground states.** If the gap vanishes, the $n_{\mathrm{occ}}$-th and $(n_{\mathrm{occ}}+1)$-th levels cross and the Slater determinant becomes ambiguous; one must then resolve the degeneracy (e.g. with a tiny symmetry-breaking field) before the FHS link variables are well defined.
- **Interaction.** The Slater-determinant construction assumes a non-interacting ground state. The *same* flux-torus FHS algorithm applies verbatim to an interacting ground state $|\Psi(\boldsymbol\theta)\rangle$ obtained by exact diagonalisation or DMRG — only the state preparation changes, which is the route to fractional Chern insulators (Sheng et al., PRL **107**, 146803 (2011); Sun et al., PRL **106**, 236803 (2011)).
